# Llama 3.2 1B Base: SFT then DPO

Recipe: start from `meta-llama/Llama-3.2-1B`, SFT on UltraChat, then DPO on UltraFeedback. The key control is `PROMPT_FORMAT = "plain"`: SFT and DPO see the same Alpaca-style prompt format, so the DPO stage is optimizing the same interface the SFT stage taught.

In [ ]:
from pathlib import Path
import os

# Kaggle: add HF_TOKEN under Add-ons > Secrets. Meta Llama models are gated,
# so your Hugging Face account must have access to the selected model.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print(f"No Kaggle HF_TOKEN secret loaded: {exc}")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])

PROJECT_DIR = Path("/kaggle/working/qwen-dpo-alignment")
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps

In [ ]:
from pathlib import Path

MODEL_NAME = "meta-llama/Llama-3.2-1B"
PROMPT_FORMAT = "plain"  # Keep SFT and DPO identical for the base model path.
DATASET_SFT = "HuggingFaceH4/ultrachat_200k"
DATASET_DPO = "HuggingFaceH4/ultrafeedback_binarized"

RUN_ROOT = Path("outputs/llama32_1b_base_sft_then_dpo")
SFT_OUT = RUN_ROOT / "sft"
DPO_OUT = RUN_ROOT / "dpo"

# Start conservative on Kaggle T4/P100. Increase these for the longer run.
SFT_TRAIN_EXAMPLES = 30000
SFT_VAL_EXAMPLES = 1000
DPO_TRAIN_EXAMPLES = 20000
DPO_VAL_EXAMPLES = 1000
MAX_LENGTH = 384
MAX_PROMPT_LENGTH = 192
SEED = 42
EVAL_STEPS = 64

## Stage 1: SFT

The 1B model can still be tight with full fine-tuning on Kaggle GPUs, so this recipe uses batch size 1, gradient checkpointing, short context, and `adamw8bit` to reduce optimizer state while keeping AdamW-style updates. For the longer run, scale `SFT_TRAIN_EXAMPLES`, `DPO_TRAIN_EXAMPLES`, `MAX_LENGTH`, or `num_epochs` after a small smoke run succeeds.

In [ ]:
!python scripts/train_sft.py \
  --model_name_or_path {MODEL_NAME} \
  --dataset_name {DATASET_SFT} \
  --train_split train_sft \
  --val_split test_sft \
  --output_dir {SFT_OUT} \
  --max_train_examples {SFT_TRAIN_EXAMPLES} \
  --max_val_examples {SFT_VAL_EXAMPLES} \
  --max_length {MAX_LENGTH} \
  --max_prompt_length {MAX_PROMPT_LENGTH} \
  --prompt_format {PROMPT_FORMAT} \
  --per_device_batch_size 1 \
  --gradient_accumulation_steps 32 \
  --num_epochs 1 \
  --learning_rate 1e-5 \
  --optimizer adamw8bit \
  --logging_steps 5 \
  --eval_steps {EVAL_STEPS} \
  --seed {SEED}

## Stage 2: DPO

Reference and policy both initialize from the SFT checkpoint. This keeps the base-model path conceptually clean: base Llama learns the instruction interface during SFT, then DPO aligns preferences on that same interface.

In [ ]:
SFT_CKPT = SFT_OUT / "best"
!python scripts/train_dpo.py \
  --model_name_or_path {SFT_CKPT} \
  --dataset_name {DATASET_DPO} \
  --train_split train \
  --val_split test \
  --output_dir {DPO_OUT} \
  --max_train_examples {DPO_TRAIN_EXAMPLES} \
  --max_val_examples {DPO_VAL_EXAMPLES} \
  --max_length {MAX_LENGTH} \
  --max_prompt_length {MAX_PROMPT_LENGTH} \
  --prompt_format {PROMPT_FORMAT} \
  --per_device_batch_size 1 \
  --gradient_accumulation_steps 32 \
  --num_epochs 1 \
  --learning_rate 5e-7 \
  --optimizer adamw8bit \
  --beta 0.1 \
  --logging_steps 5 \
  --eval_steps {EVAL_STEPS} \
  --seed {SEED}

## Training Curves

In [ ]:
!python scripts/plot_training_curves.py \
  --metrics_path {SFT_OUT}/metrics.jsonl \
  --metrics_path {DPO_OUT}/metrics.jsonl \
  --output_dir {RUN_ROOT}/plots \
  --prefix llama32_1b_base_sft_dpo

## Optional Generation For Comparison

Run this when you want pairwise comparison artifacts. The Instruct-DPO notebook has the matching generation and pair-building cell.

In [ ]:
# Optional: generate outputs for later pairwise comparison against the Instruct-DPO notebook.
# Keep prompt_format=plain because this checkpoint was SFT/DPO-trained with the plain prompt.
EVAL_DIR = RUN_ROOT / "alpaca_compare"
!python scripts/make_alpaca_subset.py --size 100 --seed {SEED} --output_dir {EVAL_DIR}
!python scripts/generate_alpaca.py \
  --model_name_or_path {DPO_OUT}/final \
  --subset_path {EVAL_DIR}/alpacaeval_100_seed{SEED}.json \
  --output_path {EVAL_DIR}/llama32_1b_base_sft_dpo_outputs.json \
  --generator_name llama32_1b_base_sft_dpo \
  --prompt_format plain \
  --batch_size 4 \
  --max_new_tokens 256 \
  --temperature 0.0